# Project: Gaussian mixtures and EM from scratch 🫧
k-means draws hard borders and assumes round clusters. A **Gaussian mixture model** gives every point a *probability* of belonging to each cluster and lets clusters be stretched ellipses. Fit it with **expectation–maximisation**: alternate soft assignments (E) and weighted re-estimation (M), and watch the log-likelihood climb every single iteration.

Topic: [[Gaussian Mixture Models and EM]], [[Clustering]] · guide note: [[Project - GMM and EM from Scratch]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
rng = np.random.default_rng(0)
true_mu = [np.array([0, 0]), np.array([4, 1]), np.array([1, 4])]
true_S = [np.array([[3, 2.6], [2.6, 3]]), np.array([[0.4, 0], [0, 0.4]]), np.array([[2.5, -1.5], [-1.5, 1.5]])]
X = np.vstack([rng.multivariate_normal(m, S, n) for m, S, n in zip(true_mu, true_S, (300, 150, 250))]); labels = np.repeat([0, 1, 2], (300, 150, 250))
plt.scatter(*X.T, s=4, c=labels); plt.title("three stretched clusters"); plt.axis("equal"); plt.show()

## 1. E-step\nResponsibilities $r_{ik} = \frac{\pi_k\,\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_j \pi_j\,\mathcal N(x_i\mid\mu_j,\Sigma_j)}$. Also return the log-likelihood $\sum_i\log\sum_k\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)$. Return `(R, loglik)`.

In [ ]:
def e_step(X, pi, mu, Sigma):
    # TODO 1: weighted densities per component, normalise rows, log-likelihood
    raise NotImplementedError  # TODO 1

## 2. M-step\nWith $N_k=\sum_i r_{ik}$: $\pi_k = N_k/n$, $\mu_k = \frac1{N_k}\sum_i r_{ik}x_i$, $\Sigma_k = \frac1{N_k}\sum_i r_{ik}(x_i-\mu_k)(x_i-\mu_k)^\top$ (+ a tiny ridge $10^{-6}I$ for stability). Return `(pi, mu, Sigma)` as arrays.

In [ ]:
def m_step(X, R):
    # TODO 2: weighted mixing proportions, means and covariances
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_em():
    Xt = np.array([[0.0, 0], [0, 2], [10, 10], [10, 12]]); R = np.array([[1, 0], [1, 0], [0, 1], [0, 1.0]])
    pi, mu, S = m_step(Xt, R); R2, ll = e_step(Xt, pi, mu, S)
    return np.allclose(pi, [0.5, 0.5]) and np.allclose(mu, [[0, 1], [10, 11]]) and np.allclose(S[0], [[1e-6, 0], [0, 1 + 1e-6]]) and np.allclose(R2, R, atol=1e-6) and np.isfinite(ll)
check("EM steps", _t_em, "Hard toy responsibilities give means (0,1) and (10,11), variance 1 along y; the E-step must give them back.")
if ready("EM steps"):
    r = np.random.default_rng(1); mu = X[r.choice(len(X), 3, replace=False)]; Sigma = np.array([np.eye(2)] * 3); pi = np.ones(3) / 3; lls = []
    for it in range(100):
        R, ll = e_step(X, pi, mu, Sigma); lls.append(ll); pi, mu, Sigma = m_step(X, R)
        if it > 1 and abs(lls[-1] - lls[-2]) < 1e-6: break
    ari_gmm = adjusted_rand_score(labels, R.argmax(1)); ari_km = adjusted_rand_score(labels, KMeans(3, n_init=10, random_state=0).fit_predict(X))
    print(f"EM converged in {len(lls)} iterations; ARI vs truth: GMM {ari_gmm:.3f}  k-means {ari_km:.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4)); ax[0].plot(lls, "o-", ms=3); ax[0].set_title("log-likelihood never decreases")
    ax[1].scatter(*X.T, s=4, c=R.argmax(1)); ax[1].scatter(*mu.T, c="r", marker="x", s=80); ax[1].set_title("GMM clusters"); ax[1].axis("equal"); plt.show()
    check("EM works", lambda: all(np.diff(lls) > -1e-6) and ari_gmm > ari_km, "The log-likelihood must be monotone and the GMM should beat k-means on these stretched clusters.")

<details><summary>▶ Solution</summary>

```python
def e_step(X, pi, mu, Sigma):
    dens = np.stack([pi[k] * multivariate_normal(mu[k], Sigma[k]).pdf(X) for k in range(len(pi))], axis=1)
    ll = float(np.log(dens.sum(1)).sum())
    return dens / dens.sum(1, keepdims=True), ll
```

```python
def m_step(X, R):
    Nk = R.sum(0)
    pi = Nk / len(X)
    mu = (R.T @ X) / Nk[:, None]
    Sigma = []
    for k in range(R.shape[1]):
        d = X - mu[k]
        Sigma.append((R[:, k, None] * d).T @ d / Nk[k] + 1e-6 * np.eye(X.shape[1]))
    return pi, mu, np.array(Sigma)
```
</details>

## Stretch goals
- Choose the number of components with BIC (`-2·loglik + params·log n`) for k = 1…6.
- Restart EM from 10 random initialisations: how often does it get stuck in a worse local optimum?
- Use the fitted GMM for anomaly detection: low-density points ([[Anomaly Detection]]).